# QAIRT 2.42 Supertonic vector/vocoder contexts

Independent offline context-generation attempts for the exact vector estimator and vocoder on SM8850 / HTP-v81. This notebook does not train, run on a phone, or prove NPU execution. Each component has separate durable state so a vector conversion failure does not prevent a vocoder attempt.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess

EXPECTED = {
    'qairt242_colab_supertonic_remaining.py': '9a18437aa5eab04a12632744af401d8cf6e17c6da633fba0dcb1ee354288e2b6',
    'qairt242_colab_supertonic_duration_canary.py': 'bc39dc8fe71206177321c5fa9165cc3d7ea7d78ef3fbdb3b7e0add5f4a802ca2',
    'qairt242_colab_supertonic_remaining_spec.json': '6a843bcb2eeb7819d0e16b81fab16b7e2ab6093db51d413e7a5e624c9b8dcc40',
}

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

print({'platform': platform.platform(), 'machine': platform.machine()})

In [ ]:
from google.colab import drive, files

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/lmf_qairt242_colab_supertonic_remaining_v1')
CONTROL = ROOT / 'control'
WORKSPACE = ROOT / 'workspace'
SCRATCH = Path('/content/qairt242_supertonic_remaining_scratch')
CONTROL.mkdir(parents=True, exist_ok=True)
WORKSPACE.mkdir(parents=True, exist_ok=True)

for name, expected_hash in EXPECTED.items():
    existing = CONTROL / name
    if existing.exists() and sha256_file(existing) != expected_hash:
        raise RuntimeError(f'Fail-closed: existing control drift: {existing}')

missing = [name for name in EXPECTED if not (CONTROL / name).is_file()]
if missing:
    print('Upload exactly these repository files:', missing)
    uploaded = files.upload()
    for name in missing:
        if name not in uploaded:
            raise RuntimeError(f'Missing uploaded control file: {name}')
        data = uploaded[name]
        if sha256_bytes(data) != EXPECTED[name]:
            raise RuntimeError(f'Fail-closed: uploaded hash mismatch: {name}')
        temporary = CONTROL / f'.{name}.partial'
        temporary.write_bytes(data)
        temporary.replace(CONTROL / name)

for name, expected_hash in EXPECTED.items():
    assert sha256_file(CONTROL / name) == expected_hash
print({'control_verified': True, 'workspace': str(WORKSPACE)})

Install the supported host toolchain in Colab only. No GPU is required.

In [ ]:
apt_env = dict(os.environ)
apt_env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['apt-get', 'update', '-qq'], check=True, env=apt_env)
subprocess.run([
    'apt-get', 'install', '-y', '--no-install-recommends',
    'python3.10', 'python3.10-venv', 'python3-distutils',
    'build-essential', 'clang-14', 'cmake', 'ninja-build',
    'curl', 'unzip', 'bzip2', 'ca-certificates'
], check=True, env=apt_env)
python310 = shutil.which('python3.10')
if not python310:
    raise RuntimeError('Python 3.10 was not installed')
subprocess.run([python310, '--version'], check=True)
subprocess.run(['clang-14', '--version'], check=True)

Review both zero-write plans. They must name QAIRT 2.42, HTP, `soc_model=87`, `dsp_arch=v81`, and only the selected graph.

In [ ]:
runner = CONTROL / 'qairt242_colab_supertonic_remaining.py'
spec = CONTROL / 'qairt242_colab_supertonic_remaining_spec.json'
for component in ('vector_estimator', 'vocoder'):
    subprocess.run([
        python310, str(runner), 'plan', '--component', component,
        '--spec', str(spec),
    ], check=True)

Run or resume both independent components. A failed vector conversion is expected to remain possible because its raw graph contains unsupported-looking operators; the cell records that failure and still attempts the vocoder.

In [ ]:
run_results = {}
for component in ('vector_estimator', 'vocoder'):
    completed = subprocess.run([
        python310, str(runner), 'run', '--component', component,
        '--spec', str(spec), '--workspace', str(WORKSPACE),
        '--scratch', str(SCRATCH),
    ], check=False)
    run_results[component] = completed.returncode
print({'component_return_codes': run_results})

Verify every completed component and print failure-report paths for failed components. Generated contexts are still unexecuted and are not NPU evidence.

In [ ]:
summary = {}
for component in ('vector_estimator', 'vocoder'):
    component_root = WORKSPACE / 'components' / component
    state_path = component_root / 'qairt242_colab_state.json'
    state = json.loads(state_path.read_text()) if state_path.is_file() else {}
    if state.get('status') == 'complete':
        subprocess.run([
            python310, str(runner), 'verify', '--component', component,
            '--spec', str(spec), '--workspace', str(WORKSPACE),
        ], check=True)
        manifest_path = component_root / 'stages/deliverable/result_manifest.json'
        manifest = json.loads(manifest_path.read_text())
        summary[component] = {
            'status': manifest['status'],
            'context_sha256': manifest['context']['context_sha256'],
            'phone_execution_proven': manifest['phone_execution_proven'],
            'npu_execution_proven': manifest['npu_execution_proven'],
        }
    else:
        summary[component] = {
            'status': state.get('status', 'not_started'),
            'failure_report': str(component_root / 'qairt242_colab_failure_report.json'),
        }
print(json.dumps(summary, indent=2, sort_keys=True))